# Edge Features — Minh hoạ kết quả từ `edges.json`
Load trực tiếp `data/preprocessed/edges.json` (đã tính trên Colab),  
tham chiếu claim text gốc từ `factoid_claims_revised.jsonl` để minh hoạ 10 edges tiêu biểu.

In [1]:
import sys, json
from pathlib import Path

PROJECT     = Path().resolve()
CLAIMS_PATH = PROJECT / "data" / "preprocessed" / "factoid_claims_revised.jsonl"
EDGES_PATH  = PROJECT / "data" / "preprocessed" / "edges.json"

sys.path.insert(0, str(PROJECT))

print("Project    :", PROJECT)
print("Claims     :", CLAIMS_PATH.exists())
print("Edges.json :", EDGES_PATH.exists())

Project    : D:\03 Data science\00 AIO2025\01 Projects\rese006-conflict-graph-agentic-rag\clone\conflict-graph-agentic-rag
Claims     : True
Edges.json : True


In [2]:
# ── Claim text lookup: claim_id → {claim_text, factoid_features} ──────────────
claim_lookup: dict = {}

with open(CLAIMS_PATH, encoding="utf-8") as f:
    for line in f:
        c = json.loads(line)
        claim_lookup[c["claim_id"]] = c

print(f"{len(claim_lookup):,} claims loaded")

10,550 claims loaded


In [3]:
# ── Load edges.json, chọn 10 edges tiêu biểu ─────────────────────────────────
# Chiến lược: lấy 2-3 edges có contradiction_prob cao nhất,
#             2-3 edges support cao nhất, phần còn lại neutral/entailment
# → minh hoạ đủ 4 loại relation.

with open(EDGES_PATH, encoding="utf-8") as f:
    all_edges = json.load(f)

print(f"Loaded {len(all_edges):,} edges từ edges.json")

def _contra(e): return e["nli_features"]["contradiction_prob"]
def _support(e): return e["nli_features"]["support_prob"]
def _entail(e):  return e["nli_features"]["entailment_prob"]

# Phân nhóm theo relation_type
by_rel: dict = {"contradiction": [], "support": [], "entailment": [], "neutral": []}
for e in all_edges:
    rel = e.get("relation_type", "neutral")
    if rel in by_rel:
        by_rel[rel].append(e)

# Sort mỗi nhóm theo confidence giảm dần
by_rel["contradiction"].sort(key=_contra,  reverse=True)
by_rel["support"].sort(     key=_support, reverse=True)
by_rel["entailment"].sort(  key=_entail,  reverse=True)
by_rel["neutral"].sort(     key=lambda e: e["relation_confidence"], reverse=True)

# Ghép: 3 contradiction, 3 support, 2 entailment, 2 neutral
sample_edges = (
    by_rel["contradiction"][:3] +
    by_rel["support"][:3]       +
    by_rel["entailment"][:2]    +
    by_rel["neutral"][:2]
)[:10]

print(f"\nSelected {len(sample_edges)} edges:")
counts = {}
for e in sample_edges:
    counts[e["relation_type"]] = counts.get(e["relation_type"], 0) + 1
for rel, n in counts.items():
    print(f"  {rel:15s}: {n}")

Loaded 96,700 edges từ edges.json

Selected 10 edges:
  contradiction  : 3
  support        : 3
  entailment     : 2
  neutral        : 2


In [4]:
# ── Summary DataFrame ─────────────────────────────────────────────────────────
import pandas as pd
pd.set_option("display.max_colwidth", 55)

rows = []
for e in sample_edges:
    nli = e["nli_features"]
    agg = e["claim_pair_features"]["aggregate"]
    src = claim_lookup.get(e["source_claim_id"], {})
    tgt = claim_lookup.get(e["target_claim_id"], {})
    rows.append({
        "relation"        : e["relation_type"],
        "confidence"      : e["relation_confidence"],
        "contra_p"        : nli["contradiction_prob"],
        "entail_p"        : nli["entailment_prob"],
        "support_p"       : nli["support_prob"],
        "neutral_p"       : nli["neutral_prob"],
        "conflict_intens" : agg["conflict_intensity_score"],
        "dominant_fail"   : agg["dominant_fail_type"],
        "src_text"        : src.get("claim_text", "")[:60],
        "tgt_text"        : tgt.get("claim_text", "")[:60],
    })

df = pd.DataFrame(rows)
df

,relation,confidence,contra_p,entail_p,support_p,neutral_p,conflict_intens,dominant_fail,src_text,tgt_text
0,contradiction,1.0000,1.0000,0.0000,0.0000,0.0000,0.7000,number_mismatch,"Retrieved September 10 , 2020 .","Retrieved January 3 , 2013 ."
1,contradiction,0.9999,0.9999,0.0000,0.0000,0.0001,0.6999,number_mismatch,"Bryan was born in Alexandria, Louisiana on February...","Bryan was born in Alexandria, Louisiana on July 15,..."
2,contradiction,0.9999,0.9999,0.0000,0.0000,0.0001,0.6999,entity_mismatch,`` Valley of Death '' 3:27 6.,`` Shine On '' 3:13 3 .
3,support,0.9986,0.0004,0.0001,0.9986,0.0010,0.0003,temporal_mismatch,It had foundation school status which allowed a deg...,Its previous status as a foundation school allowed ...
4,support,0.9977,0.0002,0.0005,0.9977,0.0016,0.0001,none,Arteaga joined Santos Laguna youth academy in 2013.,"Lukas Nmecha Biography , Age , Parents , Wife , FIF..."
5,support,0.9971,0.0004,0.0003,0.9971,0.0022,0.0003,none,"In November 1990 it had a total of 219 tanks, 187 b...","In November 1990 , it had 219 tanks , 187 being T-5..."
6,entailment,0.6655,0.0001,0.6655,0.0007,0.3336,0.0001,none,"According to the United States Census Bureau, the t...","United States Census Bureau , the township has a to..."
7,entailment,0.6654,0.0001,0.6654,0.0001,0.3344,0.0001,none,The station is owned by SummitMedia.,"The station, which launched in 1926 as KGBX, is own..."
8,neutral,0.9996,0.0002,0.0001,0.0001,0.9996,0.0001,none,Lincoln County High School (West Virginia) Lincoln ...,The support of the community is amazing though !
9,neutral,0.9996,0.0002,0.0001,0.0001,0.9996,0.0001,entity_mismatch,"Other standout tracks include ""Stardust"" and ""North...",Music of the Spheres (Mike Oldfield album) Music of...


In [5]:
# ── Chi tiết từng edge: claim text gốc + features ────────────────────────────
from IPython.display import display, HTML

RELATION_COLOR = {
    "contradiction": "#ffe0e0",
    "support"      : "#e0f5e0",
    "entailment"   : "#daeeff",
    "neutral"      : "#f5f5f5",
}

for e in sample_edges:
    nli  = e["nli_features"]
    agg  = e["claim_pair_features"]["aggregate"]
    neg  = e["claim_pair_features"]["negation"]
    ent  = e["claim_pair_features"]["entity"]
    num  = e["claim_pair_features"].get("number") or {}
    src  = claim_lookup.get(e["source_claim_id"], {})
    tgt  = claim_lookup.get(e["target_claim_id"], {})
    color = RELATION_COLOR.get(e["relation_type"], "#f5f5f5")

    num_row = (
        f"number_presence={num.get('number_presence_i')}/{num.get('number_presence_j')} &nbsp;"
        f"number_match={num.get('number_match')} &nbsp;"
        f"number_mismatch={num.get('number_mismatch')}"
    ) if num else "<i>no number features</i>"

    html = f"""
    <div style="border:1px solid #bbb; border-radius:8px; padding:14px;
                margin-bottom:16px; background:{color}; font-family:sans-serif">

      <div style="font-size:0.8em; color:#555">{e['edge_id']}</div>
      <div style="margin:4px 0">
        <span style="font-size:1.15em; font-weight:bold;
                     color:{'#c0392b' if e['relation_type']=='contradiction' else
                            '#27ae60' if e['relation_type']=='support' else
                            '#2980b9' if e['relation_type']=='entailment' else '#555'}">
          ▶ {e['relation_type'].upper()}
        </span>
        &nbsp; <span style="color:#777">confidence = {e['relation_confidence']}</span>
      </div>

      <hr style="border:none; border-top:1px solid #ccc; margin:8px 0">

      <div style="margin-bottom:6px">
        <span style="background:#fff; border:1px solid #ddd; border-radius:4px;
                     padding:1px 6px; font-size:0.78em; font-family:monospace">
          SRC &nbsp;{e['source_claim_id']}
        </span><br>
        <span style="color:#222">{src.get('claim_text','<not found>')}</span>
      </div>

      <div>
        <span style="background:#fff; border:1px solid #ddd; border-radius:4px;
                     padding:1px 6px; font-size:0.78em; font-family:monospace">
          TGT &nbsp;{e['target_claim_id']}
        </span><br>
        <span style="color:#222">{tgt.get('claim_text','<not found>')}</span>
      </div>

      <hr style="border:none; border-top:1px solid #ccc; margin:8px 0">

      <table style="font-size:0.82em; border-collapse:collapse; width:100%">
        <tr style="background:rgba(0,0,0,0.04)">
          <td style="padding:3px 8px"><b>NLI probs</b></td>
          <td>contra = <b>{nli['contradiction_prob']:.3f}</b></td>
          <td>entail = {nli['entailment_prob']:.3f}</td>
          <td>support = {nli['support_prob']:.3f}</td>
          <td>neutral = {nli['neutral_prob']:.3f}</td>
        </tr>
        <tr>
          <td style="padding:3px 8px"><b>Aggregate</b></td>
          <td>conflict_intensity = <b>{agg['conflict_intensity_score']:.3f}</b></td>
          <td>ea_alignment = {agg['ea_alignment_score']:.3f}</td>
          <td>dominant_fail = <b>{agg['dominant_fail_type']}</b></td>
          <td>diag = {agg['diagnostic_vector']}</td>
        </tr>
        <tr style="background:rgba(0,0,0,0.04)">
          <td style="padding:3px 8px"><b>Entity</b></td>
          <td>match = {ent['entity_match']:.3f}</td>
          <td>mismatch = {ent['entity_mismatch']}</td>
          <td><b>Negation</b></td>
          <td>i={neg['negation_i']} j={neg['negation_j']} mismatch={neg['negation_mismatch']}</td>
        </tr>
        <tr>
          <td style="padding:3px 8px"><b>Number</b></td>
          <td colspan="4">{num_row}</td>
        </tr>
      </table>
    </div>
    """
    display(HTML(html))